# KI in der Medizin: Chancen und Limitierungen

Ein Modell kann viele Gewebebilder richtig zuordnen und trotzdem gerade bei seltenen Fällen versagen. Wir untersuchen, wie Trainingsdaten und die Bewertung von Fehlern solche Ergebnisse beeinflussen.

Unser Versuch vergleicht vier Modelle: eine Baseline, ein Modell mit unausgewogenen Trainingsdaten sowie zwei Gegenmaßnahmen mit Klassen-Gewichten und Focal Loss. Anschließend betrachten wir mit Grad-CAM einzelne Bilder genauer. Die acht Gewebelabels stammen aus einem Lehrdatensatz; wir entwickeln kein klinisches Diagnosewerkzeug.

Der Ablauf ist immer derselbe: **Vermutung formulieren → einen Versuchsschritt ausführen → Ergebnisse lesen → die Vermutung überprüfen.** In vier Codeaufgaben ergänzt ihr Vorhersagen, Kennzahlen, Klassen-Gewichte und Focal Loss. Hinweise und Lösungen lassen sich ausklappen. Die übrigen Funktionen sind vorbereitet.

## 1. Start und Arbeitsweise

Notebook von oben nach unten bearbeiten, Modellläufe nur nach Absprache starten. Vor jedem Versuch eine Vermutung notieren und nachher mit den Ergebnissen vergleichen. Im Zweierteam Bedienung und Erklärung abwechseln. Nach einem Projektupdate den Kernel neu starten und von oben beginnen; falls noch ein Training läuft, vorher die Workshopleitung fragen.

**Training über Nacht:** VS Code und Kernel offen lassen, Laptop am Netzteil und mit offenem Deckel stehen lassen, nicht herunterfahren oder neu starten. Windows+L zum Sperren ist erlaubt. Der Energieschutz wirkt nur während eines gestarteten Trainings. Eine laufende Zelle blockiert weitere Codezellen in demselben Kernel. Eine einzelne Zelle startet keine späteren Versuche automatisch. „Alle ausführen“ kann die vier Läufe nacheinander starten, aber nur nach Freigabe durch die Leitung.

**Gespeicherter Stand:** Nach jeder abgeschlossenen Epoche wird das Modell mit Verlauf gespeichert. Nach Neustart dieselben Einstellungen und denselben Laufnamen verwenden. Die laufende Epoche kann bei einem Abbruch verloren gehen. Fehlt ein passender Modelllauf, meldet euch bei der Workshopleitung.

In [ ]:
from pathlib import Path
import sys
for basis in (Path.cwd(), *Path.cwd().parents):
    kandidaten = (basis, basis / "science-college-ki-medizin-students")
    projekt = next((p for p in kandidaten if (p / "src" / "hauptversuch.py").is_file()), None)
    if projekt is not None:
        break
else:
    raise FileNotFoundError("Den vollständigen Schüler-Projektordner öffnen.")
for ordner in (projekt / "src", projekt / "notebooks"):
    sys.path.insert(0, str(ordner))
import numpy as np
import tensorflow as tf
import hauptversuch as hv
from versuchsbetrieb import lade_histologie, modelllauf, werte_aus, kontrolliere
from versuchsbetrieb import kontrolliere_klassengewichte, kontrolliere_metriken

# Gemeinsame Einstellungen. Änderungen erfordern neue Laufnamen.
EPOCHEN = 30
TRAININGSANTEIL = 0.75
FREIE_SCHICHTEN = 3
LAUFGRUPPE = "workshop_v2"
# True erst nach Absprache. False lädt ausschließlich abgeschlossene eigene Läufe.
TRAINING_STARTEN = False  # TODO: Vor eigenem Training nach Absprache auf True setzen.

def lauf(name, x, y, **kwargs):
    return modelllauf(projekt, LAUFGRUPPE + "_" + name, x, y,
                      daten["x_val"], daten["y_val"], epochs=EPOCHEN,
                      num_unfrozen_layers=FREIE_SCHICHTEN,
                      training_erlaubt=TRAINING_STARTEN, **kwargs)


## 2. Gewebebilder und eine gemeinsame Datengrundlage

Der Datensatz `colorectal_histology` enthält 5.000 kleine Gewebeaufnahmen mit acht Labels, darunter `tumor`, `stroma` und `adipose`. Das sind unterschiedliche Gewebetypen und Hintergründe. Die Aufgabe lautet nicht einfach „Krebs oder kein Krebs“, sondern: Welche der acht Lehrklassen passt zum Bildausschnitt?

| Label | Bedeutung im Lehrdatensatz |
| --- | --- |
| `tumor` | Tumorepithel, also Tumorzellverbände |
| `stroma` | Vergleichsweise gleichförmiges Stütz- und Bindegewebe |
| `complex` | Stützgewebe mit einzelnen Tumor- oder Immunzellen |
| `lympho` | Ansammlungen von Immunzellen |
| `debris` | Gewebereste und Schleim |
| `mucosa` | Schleimhautdrüsen |
| `adipose` | Fettgewebe |
| `empty` | Hintergrund ohne Gewebe |

Auch ein Stroma-Ausschnitt kann aus dem Umfeld eines Tumors stammen. Die Labels bedeuten deshalb nicht einfach „gesund“ und „krank“. [Quelle und Bildbeispiele: Kather et al. (2016)](https://www.nature.com/articles/srep27988).

Wir teilen die Bilder vor dem Lernen auf. **Training** verändert Modellgewichte. Die **Validierung** zeigt während der Entwicklung, wie das Modell mit zurückgehaltenen Bildern zurechtkommt. **Testbilder** dienen der anschließenden Bewertung. Alle vier Modelle werden an denselben Validierungs- und Testbildern verglichen. Nach wiederholten Testvergleichen ist das eine interne Workshop-Auswertung und keine neue unabhängige klinische Prüfung.

Standardmäßig verwenden wir 3.000 Trainings-, 500 Validierungs- und 500 Testbilder. Die weiteren 1.000 Bilder sind Reserve im Trainingspool: Sie werden beim späteren Ungleichgewichtsversuch für eine andere Auswahl gleicher Größe genutzt; sie gehören weder zur Validierung noch zum Test.

Die nächste Zelle zeigt Bildbeispiele und die Klassenverteilung in den drei Teilmengen. Vergleicht bei unterschiedlich großen Teilmengen auch die Anteile, nicht nur die Balkenhöhen.

**Bevor wir weitergehen:** Welche Klassen ähneln sich optisch? Sind einzelne Labels im Ausgangstraining wesentlich seltener als andere?

<details>
<summary>Labels als Zahlen: One-Hot-Codierung und Bildwerte</summary>

Bei acht Klassen ist Label 2 der Vektor `[0, 0, 1, 0, 0, 0, 0, 0]`. Die 1 steht an Position 2, weil Python bei 0 beginnt. Für das Training wird diese Darstellung automatisch erzeugt. Die Bilder bleiben zunächst RGB-Werte von 0 bis 255; Größenanpassung und Skalierung erfolgen erst im Modell.

</details>

In [ ]:
bilder, labels, klassennamen = lade_histologie(projekt)
daten = hv.split_data(bilder, labels, subset_frac=TRAININGSANTEIL)
print("Klassen:", list(enumerate(klassennamen)))
for rolle in ("train", "val", "test"):
    print(rolle, len(daten["y_" + rolle]),
          np.bincount(daten["y_" + rolle], minlength=8))
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 4, figsize=(11, 6))
for klasse, ax in enumerate(axes.flat):
    index = np.flatnonzero(labels == klasse)[0]
    ax.imshow(bilder[index]); ax.set_title(klassennamen[klasse]); ax.axis("off")
plt.tight_layout(); plt.show()
hv.zeige_klassenverteilung(klassennamen, Training=daten["y_train"],
                              Validierung=daten["y_val"], Test=daten["y_test"])


## 3. Transfer Learning: Ein vorhandenes Netz anpassen

Ein Bildmodell von Grund auf zu trainieren benötigt viele Beispiele und Rechenzeit. **Transfer Learning** nutzt deshalb bereits gelernte Bildmerkmale für eine neue Aufgabe. Unser MobileNetV2 wurde mit ImageNet-Alltagsbildern vortrainiert. Seine Merkmale können auch für Gewebebilder nützlich sein; ob das ausreicht, müssen wir prüfen.

Das Netz besteht für unseren Versuch aus zwei Teilen. Der **vortrainierte Rumpf** erzeugt Merkmalskarten. Der **neue Klassifikationskopf** lernt daraus unsere acht Gewebeklassen. Der ursprüngliche ImageNet-Kopf wird nicht übernommen (`include_top=False`).

**Unser Modellweg:** RGB-Bild → Größe 224×224 und Skalierung → MobileNetV2-Rumpf → räumlicher Mittelwert → 512 Neuronen → acht Klassenwerte.

**Einfrieren** bedeutet: Die Gewichte einer Schicht werden beim Training nicht verändert. Hier bleibt der größte Teil des Rumpfs eingefroren. Der neue Kopf und ein kleiner letzter Teil des Rumpfs lernen gemeinsam; dieses vorsichtige Anpassen vortrainierter Gewichte heißt **Fine-Tuning**. In diesem Versuch gibt es keine vorgeschaltete getrennte Kopf-Trainingsphase. Die nächste Zelle zeigt am geladenen Modell, welche Rumpfschichten tatsächlich lernbare Gewichte haben. `FREIE_SCHICHTEN=3` bezeichnet ein Fenster der letzten drei Layerpositionen; Batch-Normalisierung bleibt eingefroren.

### Wie das Training die Gewichte verändert

**SGD (stochastischer Gradientenabstieg)** verarbeitet kleine, zufällig gemischte Bildgruppen, sogenannte Mini-Batches. Der mittlere Loss bewertet ihre Vorhersagen. Sein Gradient beschreibt die örtliche Änderungsrichtung für jedes Modellgewicht. Die Lernrate bestimmt die Schrittgröße. Vereinfacht gilt $w_{\mathrm{neu}}=w-\eta\,\nabla L(w)$: Gewichte minus Lernrate mal Gradient. Ein Schritt erfolgt pro Batch, eine Epoche ist ein Durchgang über die Trainingsbilder. Nicht jeder einzelne Schritt verbessert das Ergebnis sicher.

Bei der **Datenerweiterung** entstehen leicht gedrehte, gespiegelte oder in der Helligkeit veränderte Ansichten der Trainingsbilder. Das Label soll dabei plausibel gleich bleiben. Diese Ansichten sind keine neuen unabhängigen Gewebeproben. Validierungs- und Testbilder werden nicht erweitert.

**Vermutung:** Welche Bildmerkmale könnten sich von Alltagsfotos auf Gewebe übertragen lassen? Warum könnte es riskant sein, sofort den ganzen Rumpf neu zu trainieren?

### Baseline laden

Die Baseline verwendet das Ausgangstraining mit annähernd gleicher Bildzahl je Klasse. Mit `TRAINING_STARTEN=False` laden wir den vollständig vorbereiteten Lauf und prüfen die Meldung „30/30 Epochen“. Das kurze Laden ist kein neues Training. Die Lernkurven zeigen anschließend die gespeicherten 30 Epochen.

<details>
<summary>Ein eigenes Training starten</summary>

Nur nach Absprache: Für ein neues Training `LAUFGRUPPE` auf einen eigenen Namen setzen, etwa `"demo_montag"`, und `TRAINING_STARTEN=True` wählen. Ein unvollständiger passender Lauf wird fortgesetzt. Zum vorbereiteten Vergleich mit `LAUFGRUPPE="workshop_v2"` und `TRAINING_STARTEN=False` von oben starten, sobald kein eigener Lauf mehr läuft. Daten, Einstellungen und Laufname müssen zusammenpassen.

</details>

In [ ]:
modell1, verlauf1 = lauf("baseline", daten["x_train"], daten["y_train"])
hv.zeige_modellaufbau(modell1)


## 4. Die Baseline beurteilen

Eine einzelne Kennzahl beantwortet nicht alle Fragen. Wir betrachten zuerst, **wie das Lernen verlaufen ist**, und danach, **welche Fehler auf den Testbildern auftreten**. Die folgenden Diagramme stammen aus dem gespeicherten Lauf.

### Lernkurven: Training und Validierung vergleichen

Die waagerechte Achse zählt Epochen. Jede Grafik zeigt eine Trainings- und eine Validierungskurve. **Accuracy** zählt richtig zugeordnete Bilder. **Macro-F1** berechnet den F1-Wert je Klasse und mittelt alle acht Klassen gleichgewichtet. **Loss** ist das beim Lernen verwendete Fehlermaß.

Lest eine Grafik nach der anderen: Steigt die Leistung auch auf der Validierung? Verbessert sich nur das Training weiter, während die Validierung schlechter wird? Ein solches Muster kann auf Überanpassung hinweisen. Training und Validierung verwenden hier unterschiedliche Bilder; Datenerweiterung macht die Trainingsaufgabe zusätzlich schwieriger. Deshalb muss eine Validierungskurve nicht immer schlechter sein als die Trainingskurve.

### Testauswertung: Fehler, Klassenanteile und Kennzahlen

In der **Konfusionsmatrix** ist jede Zeile ein Referenzlabel, jede Spalte eine Vorhersage. Die Diagonale zählt richtige Zuordnungen. Eine Zahl daneben zeigt eine konkrete Verwechslung. Nutzt die Zeilensumme, wenn ihr fragt, welcher Anteil einer tatsächlichen Klasse erkannt wird.

Die **klassenweise Kalibrierung** prüft für jede feste Klasse ihre Modellwerte auf allen Testbildern. Dazu werden getrennte Intervalle gebildet. Ein Punkt zeigt den mittleren Modellwert dieser Bilder auf der x-Achse und ihren tatsächlichen Klassenanteil auf der y-Achse. Ideal wäre die Diagonale. Ein Punkt ist keine kumulierte Summe und auch keine Gesamt-Accuracy. Die Kurve muss nicht monoton steigen oder bei (1,1) enden.

Unter der Gesamtübersicht zeigen wir eine Klasse zusätzlich mit Fallzahlen. Die Beschriftung `3/5` bedeutet: Drei der fünf Bilder dieses Intervalls tragen das betrachtete Label. Bei nur einem Bild kann der beobachtete Anteil ausschließlich 0 oder 1 sein. Leere Intervalle haben keinen Punkt. Mit `KALIBRIERUNGSKLASSE` könnt ihr später eine andere der acht Klassen wählen.

**Jetzt:** Lernkurven überblicken und Aufgabe 1 bearbeiten. Kalibrierung und die Kennzahlenaufgabe besprechen wir anschließend gemeinsam.

In [ ]:
KALIBRIERUNGSKLASSE = "stroma"  # Ändert nur die Diagrammansicht, nicht das Training.
wahrscheinlichkeiten1 = werte_aus(projekt, LAUFGRUPPE + "_baseline", modell1,
                               verlauf1, daten["x_test"], daten["y_test"], klassennamen, kalibrierungsklasse=KALIBRIERUNGSKLASSE)


### Aufgabe 1: Viele Vorhersagen gleichzeitig

Jede Tabellenzeile gehört zu einem Bild. Die drei Spalten sind drei Übungsklassen. `axis=1` bedeutet: in jeder Zeile über die Klassen suchen. Bestimme je Bild die gewählte Klasse, ihre Konfidenz und ob sie zum Label passt.

Ergänze die `TODO`-Zeilen in der nächsten Zelle und prüfe die Rückmeldungen.

In [ ]:
beispiel = np.array([[0.1, 0.7, 0.2], [0.8, 0.1, 0.1]])
wahre_labels = np.array([1, 2])
klassen = None       # TODO: Position des größten Werts in jeder Zeile
konfidenzen = None   # TODO: Größter Wert in jeder Zeile
richtig = None      # TODO: Klassen mit wahren Labels vergleichen
kontrolliere("1a Klassen", klassen, np.array([1, 0]))
kontrolliere("1b Konfidenzen", konfidenzen, np.array([0.7, 0.8]))
kontrolliere("1c Korrektheit", richtig, np.array([True, False]))


<details>
<summary>Hinweis zur Idee</summary>

Mit `argmax`, `max` und `==` bestimmst du Klasse, Konfidenz und Korrektheit. Wende die Operationen zeilenweise auf die Bilder an. Zum Nachschlagen: Grundlagen-Notebook, Abschnitt 5 / Code-Challenge 3, oder Python-/ML-Kompass „Vorhersagen prüfen“.

</details>

<details>
<summary>Hinweis zur Python-Syntax</summary>

`np.argmax(tabelle, axis=1)` liefert die Positionen; `np.max(tabelle, axis=1)` die Werte. Mit `==` kannst du die beiden Label-Arrays direkt vergleichen.

</details>

<details>
<summary>Lösung und Erklärung</summary>

```python
beispiel = np.array([[0.1, 0.7, 0.2], [0.8, 0.1, 0.1]])
wahre_labels = np.array([1, 2])
klassen = np.argmax(beispiel, axis=1)  # TODO: Position des größten Werts in jeder Zeile
konfidenzen = np.max(beispiel, axis=1) # TODO: Größter Wert in jeder Zeile
richtig = klassen == wahre_labels     # TODO: Klassen mit wahren Labels vergleichen
kontrolliere("1a Klassen", klassen, np.array([1, 0]))
kontrolliere("1b Konfidenzen", konfidenzen, np.array([0.7, 0.8]))
kontrolliere("1c Korrektheit", richtig, np.array([True, False]))
```

Für Bild 2 ist die Konfidenz 0,8, aber die Vorhersage falsch. Hohe Konfidenz garantiert keine Korrektheit.

</details>

### Aufgabe 2: Kennzahlen selbst berechnen

Übertrage die Formeln aus Grundlagen-Aufgabe 5 auf die echte Baseline. Die Konfusionsmatrix zählt acht Klassen. TP, FP und FN werden für jede Klasse bereitgestellt. Accuracy zählt alle richtigen Bilder. Precision = TP / (TP + FP), Recall = TP / (TP + FN). F1 verbindet Precision und Recall je Klasse. Macro-F1 ist der Mittelwert der acht Klassen-F1-Werte. Mit np.sum summierst du ein Array, mit np.mean bildest du seinen Mittelwert. Die bereitgestellte Funktion sicher_teilen(Zähler, Nenner) setzt ein Ergebnis bei Nenner 0 auf 0. Nach fünf richtigen Rückmeldungen erscheint die Klassentabelle mit euren berechneten Werten. Recall entspricht der Sensitivität für die jeweils betrachtete Lehrklasse. Spezifität steht als bereitgestellte Zusatzkennzahl in der Tabelle.

Ergänze die `TODO`-Zeilen in der nächsten Zelle und prüfe die Rückmeldungen.

In [ ]:
# Die Zähltabelle und die Zählgrößen sind vorbereitet.
from sklearn.metrics import confusion_matrix
vorhergesagte_labels = np.argmax(wahrscheinlichkeiten1, axis=1)
konfusionsmatrix = confusion_matrix(
    daten["y_test"], vorhergesagte_labels, labels=np.arange(8)
)
tp = np.diag(konfusionsmatrix)
fp = np.sum(konfusionsmatrix, axis=0) - tp
fn = np.sum(konfusionsmatrix, axis=1) - tp
anzahl = np.sum(konfusionsmatrix)
sicher_teilen = hv.sicher_teilen

accuracy = None   # TODO: Anteil aller richtig klassifizierten Bilder
precision = None  # TODO: Je Klasse richtige Treffer unter allen Meldungen
recall = None     # TODO: Je Klasse gefundene unter allen tatsächlichen Fällen
f1 = None         # TODO: Precision und Recall je Klasse harmonisch verbinden
macro_f1 = None   # TODO: Die acht F1-Werte gleichgewichtet mitteln

kontrolliere_metriken(
    accuracy, precision, recall, f1, macro_f1, konfusionsmatrix, klassennamen
)


<details>
<summary>Hinweis zur Idee</summary>

Die Diagonale zählt alle richtigen Vorhersagen. Bei Precision beziehen wir uns auf gemeldete Fälle, bei Recall auf tatsächlich vorhandene Fälle. Nimm für F1 zuerst die Perspektive jeder einzelnen Klasse ein.

</details>

<details>
<summary>Hinweis zur Python-Syntax</summary>

sicher_teilen(zaehler, nenner) funktioniert für alle acht Einträge gleichzeitig. np.sum(tp) zählt Treffer über alle Klassen. np.mean(f1) bildet den Mittelwert.

</details>

<details>
<summary>Lösung und Erklärung</summary>

```python
# Die Zähltabelle und die Zählgrößen sind vorbereitet.
from sklearn.metrics import confusion_matrix
vorhergesagte_labels = np.argmax(wahrscheinlichkeiten1, axis=1)
konfusionsmatrix = confusion_matrix(
    daten["y_test"], vorhergesagte_labels, labels=np.arange(8)
)
tp = np.diag(konfusionsmatrix)
fp = np.sum(konfusionsmatrix, axis=0) - tp
fn = np.sum(konfusionsmatrix, axis=1) - tp
anzahl = np.sum(konfusionsmatrix)
sicher_teilen = hv.sicher_teilen

accuracy = np.sum(tp) / anzahl   # TODO: Anteil aller richtig klassifizierten Bilder
precision = sicher_teilen(tp, tp + fp)  # TODO: Je Klasse richtige Treffer unter allen Meldungen
recall = sicher_teilen(tp, tp + fn)     # TODO: Je Klasse gefundene unter allen tatsächlichen Fällen
f1 = sicher_teilen(2 * precision * recall, precision + recall)         # TODO: Precision und Recall je Klasse harmonisch verbinden
macro_f1 = np.mean(f1)   # TODO: Die acht F1-Werte gleichgewichtet mitteln

kontrolliere_metriken(
    accuracy, precision, recall, f1, macro_f1, konfusionsmatrix, klassennamen
)
```

Macro-F1 mittelt die acht F1-Werte. Ein harmonisches Mittel aus Macro-Precision und Macro-Recall ergibt im Allgemeinen eine andere Zahl. Die Testkennzahlen werden ohne die Trainings-Klassengewichte berechnet. Alle Werte liegen zwischen 0 und 1. Eine Klasse ohne Treffer kann einen Recall von 0 haben, obwohl Accuracy hoch ist.

</details>

### Baseline: Beobachtungen festhalten

**Gemeinsam bzw. zu zweit:**
1. Wie unterscheiden sich Training und Validierung im Verlauf? Nennt einen konkreten Abschnitt der Lernkurven.
2. Welche Verwechslung fällt in der Konfusionsmatrix auf? Nennt Referenzklasse, Vorhersageklasse und Bildzahl.
3. Wie unterscheiden sich Accuracy und Macro-F1? Welche Klassenwerte helfen, den Unterschied zu erklären?

Unser Testsatz enthält annähernd gleich viele Bilder je Klasse. Accuracy und Macro-F1 können deshalb nahe beieinanderliegen; trotzdem können einzelne Klassen deutlich schlechter erkannt werden. Prüft dafür die Klassentabelle.

<details>
<summary>Vertiefung: Kalibrierung und weitere Prüffragen</summary>

Wählt einen Punkt der Detailkurve. Was bedeuten seine beiden Koordinaten und wie viele Bilder stehen dahinter? Was bedeutet seine Lage zur Diagonale? Würde ein einzelnes anderes Label den Punkt stark verändern?

Ein zackiger Verlauf allein beweist weder zu wenig Training noch eine stabile Fehlkalibrierung. Ein hoher Gesamtwert kann außerdem Fehler einer einzelnen Klasse verdecken. Haltet eine Aussage fest, die eure Daten stützen, und eine Frage, die offenbleibt.

</details>

## 5. Klassenungleichgewicht gezielt untersuchen

Medizinische Datensätze bilden nicht automatisch alle Fälle gleich häufig ab. Seltene Gewebetypen liefern weniger Trainingsbeispiele; verschiedene Kliniken können außerdem andere Untersuchungen, Geräte oder Patientengruppen beisteuern. Ein Modell sieht dann möglicherweise viele typische Fälle und nur wenige Beispiele der Fälle, die später trotzdem erkannt werden sollen.

Wir erzeugen ein Ungleichgewicht künstlich, damit wir seine Folgen an einer bekannten Datengrundlage untersuchen können. Das ist ein kontrolliertes Lehrbeispiel. Es bildet weder die tatsächliche Krankheitshäufigkeit einer Klinik noch Unterschiede zwischen Geschlechtern oder ethnischen Gruppen ab.

**Zuerst verändern wir die Trainingsdaten und betrachten ihre Verteilung. Erst danach laden wir das damit trainierte Modell.** Validierungs- und Testbilder bleiben gleich. So können wir die Auswertung auf denselben Bildern vergleichen.

Wir wählen aus einem gemeinsamen Trainingspool zwei gleich große Trainingssätze mit je 3.000 Bildern: die annähernd ausgewogene Baseline und eine ungleich verteilte Auswahl. Eine vorbereitete Zufallsmethode entfernt Beispiele mehrerer Klassen unterschiedlich stark; mindestens 50 Bilder pro Klasse bleiben erhalten. Ihr müsst diese Auswahlmethode nicht programmieren. Die Gesamtmenge bleibt gleich, die Zusammensetzung ändert sich. Klassen-Gewichte und Focal Loss verwenden anschließend genau denselben unausgewogenen Trainingssatz. Die Auswahl legt die seltene Klasse fest — nicht eine vorher gewünschte Modellrangfolge.

In [ ]:
pool_ids = daten["ids_pool"]
ALPHA_UNGLEICHGEWICHT = 0.1
x_schief, y_schief = hv.make_imbalanced_equal_size(
    bilder[pool_ids], labels[pool_ids], dataset_length=len(daten["y_train"]),
    alpha=ALPHA_UNGLEICHGEWICHT)
SELTENE_KLASSE = int(np.argmin(np.bincount(y_schief, minlength=8)))
SELTENES_LABEL = klassennamen[SELTENE_KLASSE]
print("Kleinste Klasse in dieser Auswahl:", SELTENES_LABEL)
hv.zeige_klassenverteilung(klassennamen, Baseline=daten["y_train"], Unausgewogen=y_schief)


### Vor dem zweiten Modelllauf

Lest die Verteilungsbalken und nennt die tatsächlichen Bildzahlen. Welche Klassen sind jetzt unterrepräsentiert? Ist die Gesamtzahl der Trainingsbilder gleich geblieben? Notiert eine Vermutung: Welche Zeile der Konfusionsmatrix könnte sich verändern, und was erwartet ihr für Recall und Macro-F1?

Die nächste Zelle lädt das Modell, das mit genau diesen Daten und Einstellungen trainiert wurde. Ein eigener veränderter Datensatz benötigt einen neuen passenden Trainingslauf; ein gespeichertes Ergebnis darf nicht als Ergebnis neuer Einstellungen ausgegeben werden.

In [ ]:
modell2, verlauf2 = lauf("unausgewogen", x_schief, y_schief)
wahrscheinlichkeiten2 = werte_aus(projekt, LAUFGRUPPE + "_unausgewogen", modell2,
                               verlauf2, daten["x_test"], daten["y_test"], klassennamen, kalibrierungsklasse=KALIBRIERUNGSKLASSE)


### Was hat das Ungleichgewicht verändert?

1. Vergleicht die seltene Klasse mit der Baseline: Welche Verwechslung und welcher Recall haben sich verändert?
2. Wie verändern sich Accuracy und Macro-F1? Erklärt einen Unterschied anhand der Klassentabelle statt nur die größere Zahl zu nennen.

<details>
<summary>Vertiefung: Lernkurven, Kalibrierung und Übertragbarkeit</summary>

Vergleicht die Lernkurven und dieselbe Klasse in den Kalibrierdiagrammen. Ist das Modell auf seltenen Fällen falsch und zugleich selbstsicher? Welche Fallzahlen stehen hinter auffälligen Punkten?

Ein einzelner Lauf erklärt noch nicht, wie viel der Änderung auf Zufall, Datenmenge oder Zusammensetzung zurückgeht. Welcher weitere Vergleich würde eure Erklärung prüfen? Unsere Testverteilung bleibt fest; eine Klinik mit anderer Verteilung ist damit noch nicht untersucht.

</details>

## 6. Gegenmaßnahme: Seltene Klassen stärker gewichten

Häufige Klassen liefern mehr Beiträge zum Trainings-Loss. **Klassen-Gewichte** verändern deshalb, wie stark der Fehler eines einzelnen Beispiels beim Lernen zählt. Ein Beispiel einer seltenen Klasse erhält hier ein höheres Gewicht. Die Bilder selbst und ihre Labels bleiben unverändert; Gewichtung erzeugt keine zusätzlichen seltenen Fälle.

Wir verwenden $w_k=N/(K\,n_k)$: Gesamtzahl $N$, Klassenanzahl $K$ und Bildzahl $n_k$ der Klasse. Bei 100 häufigen und 50 seltenen Bildern ergeben sich Gewichte von 0,75 und 1,5. Ein seltenes Beispiel zählt damit doppelt so stark. Aufgabe 3 überträgt diese Rechnung auf acht Klassen.

**Vermutung:** Könnte ein höheres Gewicht mehr seltene Fälle erfassen, dabei aber auch mehr falsche Alarme erzeugen? Nennt eine Kennzahl für beide Seiten. Nach der Aufgabe laden wir das gewichtete Modell auf denselben unausgewogenen Trainingsbildern. Architektur und Testbilder bleiben gleich.

### Aufgabe 3: Seltene Klassen stärker gewichten

Für Klasse k verwenden wir Gewicht = Gesamtzahl / (Klassenanzahl × Häufigkeit_k). Bei 150 Bildern und Häufigkeiten `[100, 50]` ergibt das `[0.75, 1.5]`. Die seltene Klasse zählt pro Beispiel doppelt so stark. Berechne jetzt die Gewichte für unsere acht Klassen. NumPy rechnet die Division für jeden Eintrag von `haeufigkeiten` einzeln.

Ergänze die `TODO`-Zeilen in der nächsten Zelle und prüfe die Rückmeldungen.

In [ ]:
haeufigkeiten = np.bincount(y_schief, minlength=8)
gesamt = len(y_schief)
gewichte = None  # TODO: Gesamtzahl / (8 * Häufigkeiten)
kontrolliere_klassengewichte(gewichte, y_schief)


<details>
<summary>Hinweis zur Idee</summary>

Setze die bereitgestellten Größen in die Formel ein. Je kleiner die Häufigkeit im Nenner, desto größer wird das Gewicht.

</details>

<details>
<summary>Hinweis zur Python-Syntax</summary>

`gesamt / (8 * haeufigkeiten)` arbeitet mit dem ganzen Zahlenarray.

</details>

<details>
<summary>Lösung und Erklärung</summary>

```python
haeufigkeiten = np.bincount(y_schief, minlength=8)
gesamt = len(y_schief)
gewichte = gesamt / (8 * haeufigkeiten)  # TODO: Gesamtzahl / (8 * Häufigkeiten)
kontrolliere_klassengewichte(gewichte, y_schief)
```

`np.bincount` zählt, wie viele Trainingsbilder zu jeder Klasse gehören. Die Division berechnet für jede Klasse ein eigenes Gewicht.

</details>

In [ ]:
klassengewichte = hv.class_weights(y_schief, num_classes=8)
print(klassengewichte)
modell3, verlauf3 = lauf("gewichtet", x_schief, y_schief, class_weight=klassengewichte)
wahrscheinlichkeiten3 = werte_aus(projekt, LAUFGRUPPE + "_gewichtet", modell3,
                               verlauf3, daten["x_test"], daten["y_test"], klassennamen, kalibrierungsklasse=KALIBRIERUNGSKLASSE)


### Hilft die Gewichtung — und wem?

1. Verbessert sich der Recall der seltenen Klasse gegenüber dem ungewichteten Lauf? Was passiert gleichzeitig mit ihrer Precision oder mit Fehlern anderer Klassen?
2. Warum kann ein höheres Gewicht fehlende Bildvarianten einer Klasse nicht ersetzen? Bezieht euch auf die unveränderten Trainingsbilder.

<details>
<summary>Vertiefung: Die Losskurven richtig vergleichen</summary>

Beim gewichteten Modell ist der Trainings-Loss mit Klassen-Gewichten berechnet, der Validierungs-Loss hier ungewichtet. Ihre absolute Höhe ist deshalb nicht unmittelbar vergleichbar. Eine Niveau-Lücke allein ist kein Overfitting-Nachweis. Betrachtet zusätzlich die Verläufe sowie Accuracy und Macro-F1 auf der Validierung.

Prüft außerdem dieselbe Klasse in den Kalibrierdiagrammen. Bessere Trefferzahlen bedeuten nicht automatisch zuverlässigere Wahrscheinlichkeitswerte oder einen Fairnessnachweis.

</details>

## 7. Gegenmaßnahme: Mit Focal Loss anders aus Fehlern lernen

Klassen-Gewichte unterscheiden häufige und seltene Klassen. **Focal Loss** berücksichtigt zusätzlich, wie leicht ein einzelnes Beispiel bereits erkannt wird. Viele leichte Beispiele sollen das Lernen weniger dominieren; schwierig eingeordnete Beispiele behalten relativ mehr Einfluss.

Aus den Grundlagen kennen wir die Kreuzentropie $-\ln(p_t)$. Dabei ist $p_t$ der Modellwert für das **wahre Label**, nicht automatisch der größte Wert. Focal Loss ergänzt das Klassen-Gewicht $\alpha_t$ und den Fokusfaktor $(1-p_t)^\gamma$:

$$L_{\mathrm{focal}}=-\alpha_t(1-p_t)^\gamma\ln(p_t).$$

Für $\gamma=2$ ist der Fokusfaktor bei $p_t=0{,}95$ nur 0,0025, bei $p_t=0{,}40$ dagegen 0,36. Der erste Fehlerbeitrag wird also viel stärker gedämpft. **Modell 4 verwendet dieselben Klassen-Gewichte wie Modell 3 und ergänzt diesen Fokusfaktor.** Bei $\gamma=0$ bleibt die gewichtete Kreuzentropie.

Focal Loss garantiert keine Verbesserung: Auch falsch gelabelte oder ungewöhnliche Bilder können schwierig sein und relativ mehr Einfluss erhalten. Notiert vor Aufgabe 4, welche Veränderung ihr erwartet. Danach vergleichen wir alle vier Modelle auf denselben Testbildern.

### Aufgabe 4: Focal Loss aus bekannten Bausteinen

Wir verwenden wie in den Grundlagen genau eine Vorhersage: `y_true` ist der One-Hot-Vektor der wahren Klasse. Berechne `p_t`, `alpha_t` und `-alpha_t * (1-p_t)**gamma * log(p_t)`. Die spätere Verarbeitung vieler Bilder übernimmt der vorbereitete Trainingscode.

Ergänze die `TODO`-Zeilen in der nächsten Zelle und prüfe die Rückmeldungen.

In [ ]:
y_true = tf.constant([0., 1., 0.])
y_pred = tf.constant([0.1, 0.6, 0.3])
alpha = tf.constant([1., 2., 1.])
gamma = 2.
p_t = None        # TODO: Wahrscheinlichkeit der wahren Klasse auswählen und summieren
alpha_t = None    # TODO: Gewicht der wahren Klasse auswählen und summieren
focal_wert = None # TODO: Focal-Loss-Formel einsetzen
kontrolliere("4a p_t", p_t, .6)
kontrolliere("4b alpha_t", alpha_t, 2.)
kontrolliere("4c Focal Loss", focal_wert, -2 * (1-.6)**2 * np.log(.6))


<details>
<summary>Hinweis zur Idee</summary>

Die 1 im One-Hot-Vektor wählt den passenden Eintrag aus. Die anderen Einträge werden bei der Multiplikation null. Danach summierst du. Zum Nachschlagen: Grundlagen-Notebook, Abschnitt 7 / Code-Challenge 6, oder Python-/ML-Kompass „Loss verstehen“.

</details>

<details>
<summary>Hinweis zur Python-Syntax</summary>

`tf.reduce_sum(a * b)` verbindet Auswahl und Summe. `**` potenziert, `tf.math.log` berechnet den natürlichen Logarithmus.

</details>

<details>
<summary>Lösung und Erklärung</summary>

```python
y_true = tf.constant([0., 1., 0.])
y_pred = tf.constant([0.1, 0.6, 0.3])
alpha = tf.constant([1., 2., 1.])
gamma = 2.
p_t = tf.reduce_sum(y_true * y_pred)  # TODO: Wahrscheinlichkeit der wahren Klasse auswählen und summieren
alpha_t = tf.reduce_sum(y_true * alpha) # TODO: Gewicht der wahren Klasse auswählen und summieren
focal_wert = -alpha_t * (1-p_t)**gamma * tf.math.log(p_t) # TODO: Focal-Loss-Formel einsetzen
kontrolliere("4a p_t", p_t, .6)
kontrolliere("4b alpha_t", alpha_t, 2.)
kontrolliere("4c Focal Loss", focal_wert, -2 * (1-.6)**2 * np.log(.6))
```

Hier ist die wahre Klasse an Position 1: p_t=0,6 und alpha_t=2. Der Faktor (1-p_t)^gamma dämpft den Loss bei großen p_t stärker.

</details>

In [ ]:
alpha_modell = np.array([klassengewichte[i] for i in range(8)], dtype="float32")
modell4, verlauf4 = lauf("focal", x_schief, y_schief, loss="focal", alpha=alpha_modell, gamma=2.)
wahrscheinlichkeiten4 = werte_aus(projekt, LAUFGRUPPE + "_focal", modell4,
                               verlauf4, daten["x_test"], daten["y_test"], klassennamen, kalibrierungsklasse=KALIBRIERUNGSKLASSE)
vergleich = hv.zeige_modellvergleich(
    daten["y_test"],
    {"Baseline": wahrscheinlichkeiten1, "Unausgewogen": wahrscheinlichkeiten2,
     "Gewichtet": wahrscheinlichkeiten3, "Focal Loss": wahrscheinlichkeiten4},
    klassennamen, zielklasse=SELTENES_LABEL
)


### Vier Modelle: Welche Verbesserung ist tatsächlich zu sehen?

1. Vergleicht Focal Loss mit dem gewichteten Modell: Welche Kennzahl der seltenen Klasse verbessert oder verschlechtert sich? Nennt zusätzlich Accuracy oder Macro-F1 aus der gemeinsamen Tabelle.
2. Würdet ihr für alle Klassen dieselbe Rangfolge der Modelle erhalten? Belegt eure Antwort mit mindestens einer weiteren Klasse.

<details>
<summary>Vertiefung: Unterschiedliche Lossfunktionen und Kalibrierung</summary>

Vergleicht die Lernverläufe und ausgewählte Punkte der Kalibrierkurven. Absolute Losswerte von Kreuzentropie, gewichteter Kreuzentropie und Focal Loss sind keine gemeinsame Güteskala. Ein kleinerer Focal-Loss-Wert bedeutet daher nicht automatisch ein besseres Modell. Die ungewichteten Testkennzahlen verwenden dagegen für alle vier Modelle dieselben Labels und Definitionen.

Eine Gegenmaßnahme kann Fehler verschieben, ohne fehlende Datenvielfalt zu ersetzen. Welche zusätzliche Untersuchung wäre nötig, bevor man eine Verbesserung auf andere Daten überträgt?

</details>

## 8. Grad-CAM: Bildbereiche für eine Klassenentscheidung untersuchen

Kennzahlen zeigen, wie häufig ein Modell richtig liegt. Sie verraten jedoch nicht, welche Bildinformationen es nutzt. Ein Modell könnte beispielsweise ein hilfreiches Gewebemuster erkennen oder mit einem zufälligen Hintergrundmerkmal arbeiten. Deshalb betrachten wir nun einzelne Vorhersagen genauer.

**Wechsel zur erklärbaren KI (XAI):** Klassen-Gewichte und Focal Loss verändern das Training. Grad-CAM untersucht anschließend die bereits trainierten Modelle. Deren Modellgewichte bleiben dabei unverändert.

**Grad-CAM** steht für *Gradient-weighted Class Activation Mapping*. Eine späte Faltungsschicht des Netzes erzeugt Merkmalskarten: Sie beschreiben gelernte Muster an verschiedenen Bildpositionen. Für eine ausgewählte **Zielklasse** berechnet Grad-CAM, wie empfindlich deren Modellwert auf diese Merkmalskarten reagiert. Die Karten werden entsprechend gewichtet und zu einer groben räumlichen Darstellung zusammengeführt. Diese wird vergrößert und als Farbkarte über das Bild gelegt.

Die hervorgehobenen Regionen sind ein Hinweis darauf, welche Bereiche die Entscheidung für diese Klasse unterstützen könnten. Das ist kein direkt gemessener Blick und kein Beweis einer Ursache. Die Karte ist räumlich grob; sie zeigt keine medizinisch bestätigten Zellgrenzen.

### Zwei Fälle gezielt vergleichen

Mit `FALL="case1"` suchen wir ein Bild, das nur Modell 2 falsch zuordnet. Mit `FALL="case2"` suchen wir ein Bild, das alle Modelle richtig zuordnen, bei dem sich ihre Konfidenzen unter den gemeinsam richtig erkannten Bildern möglichst stark unterscheiden. Ob solche Bilder vorhanden sind, ist ein Messergebnis. Wird ein Ersatzbild gemeldet, erfüllt es den gesuchten Filter nicht.

Für jedes Bild betrachten alle vier Heatmaps dieselbe Zielklasse, hier das Referenzlabel. Falls Modell 2 eine andere Klasse vorhersagt, fragt seine Karte trotzdem nach dem Beitrag zur gemeinsamen Zielklasse. Wir vergleichen damit dieselbe Frage an vier Modelle. Jede Karte wird separat skaliert: Vergleicht die Lage der markierten Bereiche, nicht ihre absolute Helligkeit.

**Vermutung vor dem Ausführen:** Erwarten wir am selben Bild dieselben Markierungen, wenn die Modelle unterschiedliche Klassenwerte liefern? Führt zuerst Fall 1 aus und wiederholt anschließend mit Fall 2.

<details>
<summary>Originalpaper und unsere Umsetzung</summary>

Die Abbildungen 1 und 2 stammen aus Selvaraju et al., [Grad-CAM, arXiv v4 (2019)](https://arxiv.org/abs/1610.02391v4). Die PDF liegt im Projektordner `Hintergrund`. Guided Grad-CAM ist ein zusätzlicher Zweig der Originalabbildung und wird hier nicht verwendet. Das Paper berechnet bei Klassifikation Gradienten des Klassenwerts vor Softmax, also vor der Umrechnung in Wahrscheinlichkeiten. Unser bestehender Workshopcode verwendet die ausgegebene Klassenwahrscheinlichkeit. Beide folgen der erklärten Grundidee, die Zielwerte sind jedoch verschieden.

</details>

In [ ]:
vorhersagen = [wahrscheinlichkeiten1, wahrscheinlichkeiten2, wahrscheinlichkeiten3, wahrscheinlichkeiten4]
auswahl = hv.choose_examples(vorhersagen, daten["y_test"])
for fall in ("case1", "case2"):
    print(fall, auswahl[fall][:5])
    if auswahl[fall + "_missing"]:
        print("Kein passendes Beispiel. Ersatz ist ausdrücklich kein Treffer dieses Filters.")
FALL = "case1"  # Danach "case2" wählen und diese Zelle erneut ausführen.
if FALL not in ("case1", "case2"):
    raise ValueError('FALL muss "case1" oder "case2" sein.')
beispiel_index = int(auswahl[FALL][0]) if len(auswahl[FALL]) else auswahl["fallback"]
print("Untersuchtes Bild:", beispiel_index)
for nummer, modell in enumerate((modell1, modell2, modell3, modell4), 1):
    print("Modell", nummer)
    hv.plot_example(modell, daten["x_test"][beispiel_index],
                    int(daten["y_test"][beispiel_index]), klassennamen,
                    target_class=int(daten["y_test"][beispiel_index]))


### Beobachtung und Erklärung auseinanderhalten

1. Nennt Bildindex, Referenzlabel, Vorhersage und Konfidenz. Was unterscheidet sich zwischen den vier Modellen?
2. Wo liegen die hervorgehobenen Bereiche? Welche Beobachtung passt zu eurer Vermutung, welche nicht?
3. Formuliert eine mögliche Erklärung und einen nächsten Test. Ein gezielt veränderter Bildausschnitt könnte beispielsweise eine Hypothese prüfen; eine Heatmap allein entscheidet sie nicht.

Ein Modell mit günstigerer Lossfunktion muss nicht automatisch eine „bessere“ Heatmap erzeugen. Auch richtig zugeordnete Bilder können fragwürdige Muster zeigen. Begründet eure Aussagen am Bild und vermeidet Aussagen wie „Die KI hat Krebs verstanden“.

## 9. Unsere Beobachtung für den Science Slam

Haltet eine Beobachtung und eine offene Frage fest. Welche Idee könnte man einem Publikum mit einem kurzen Beispiel erklären? Der gemeinsame Science Slam dauert insgesamt zehn Minuten. Themen und Arbeitsauftrag stehen in [README – Science Slam](../README.md#science-slam).

<details>
<summary>Bereitgestellten technischen Code ansehen</summary>

Datenaufteilung, Metriken, Modellbau und Grad-CAM stehen in `src/hauptversuch.py`. Lokale Trainingsstände verwaltet `src/versuchsbetrieb.py`. Diese Funktionen müsst ihr nicht verändern.

</details>